# Discrete-event experiments — analysis & visualisation

This notebook reads the **archived simulation data** written by `discrete_events.py`
and reproduces everything that used to be baked into the run script:

* **divisions** / **deaths** (flat square) — a colour-coded 2-D **GIF** (faces by
  cell state: grey `normal`, magenta `dividing`, black `extruding`) + still frames.
* **gillespie** (3-D crypt) — the cell-type colour-coded 3-D **GIF** + stills, and
  three analyses: cell-type distribution over time, cell-type spatial distribution
  along z (crypt length), and event-type (division / differentiation / extrusion)
  spatial distribution along z.

Each scenario's `outputs/<scenario>/history.hf5` is reopened with
`HistoryHdf5.from_archive`; the gillespie **events** (which come from the process
emitter, not the History) are read from `outputs/gillespie/events.csv`. Re-analyse
without re-simulating, and re-run `discrete_events.py` without disturbing earlier
analysis.

Run from the repo's **`vivarium-tyssue` conda env** (needs ImageMagick `magick`
on PATH for the GIFs).

In [ ]:
from __future__ import annotations
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

# Simulation config is the single source of truth — import it from the sim script.
import discrete_events as sim
from discrete_events import OUT_DIR, COORDS_2D, COORDS_3D

sys.path.insert(0, str(sim.REPO))   # so vivarium_tyssue.draw / tyssue import cleanly
from tyssue.core.history import HistoryHdf5

# --- visualisation / analysis constants (were in the old monolithic script) ---
FIG_DPI = 300          # publication-quality raster resolution for stills / plots
GIF_DPI = 120          # animations (kept lower — many frames)
NUM_GIF_FRAMES = 120
N_STILLS = 5

# Flat-sheet cell "states": neutral background + the two event highlights.
FLAT_TYPE_COLORS = {"normal": "#CFCFCF", "dividing": "#C71FE0", "extruding": "#000000"}

Z_NBINS = 12       # event-type histogram along z
Z_NBINS_CT = 48    # finer bins for the cell-type-along-z line plot

BIO_TYPES = ["sc", "pc", "ent", "gc"]                       # true crypt cell types
CELL_TYPE_ORDER = BIO_TYPES + ["dividing", "extruding"]     # + transient states

print("archives:", OUT_DIR)

## Loading an archived history

Same helper as the other experiment notebooks: reopen the archive with
`HistoryHdf5.from_archive` for drawing, read the full stacked dataframes from the
HDF5 store for analysis, and rebuild each retrieved frame from those stacked tables
(restoring the per-frame topological index from the `vert` / `edge` / `face`
columns) so drawing and geometry updates line up.

In [ ]:
class LoadedHistory:
    # Reopened archive with the same surface as an in-memory tyssue History:
    # .time_stamps, .retrieve(t) (drawing), .datasets (full stacked dfs, analysis).
    def __init__(self, path):
        self._h = HistoryHdf5.from_archive(str(path))
        self._sheet = self._h.sheet
        with pd.HDFStore(str(path), "r") as store:
            self.datasets = {k.strip("/"): store.select(k) for k in store.keys()}
        self._times = np.array(sorted(self.datasets["vert"]["time"].unique()))

    @property
    def time_stamps(self):
        return self._times

    def retrieve(self, t):
        # Rebuild a sheet at the nearest recorded time from the stacked datasets,
        # restoring each element's per-frame index from its vert/edge/face column.
        t = self._times[int(np.argmin(np.abs(self._times - t)))]
        sheet_datasets = {}
        for elem, df in self.datasets.items():
            sub = df[df["time"] == t]
            if elem in sub.columns:
                sub = sub.set_index(elem)
                sub.index.name = elem
            sheet_datasets[elem] = sub
        sheet = type(self._sheet)(f"{self._sheet.identifier}_{t:04.3f}",
                                  sheet_datasets, self._sheet.specs)
        sheet.coords = self._sheet.coords
        return sheet

    def update_datasets(self):
        pass

    def __getattr__(self, name):
        return getattr(self._h, name)


def load_events(path) -> list:
    # gillespie events.csv -> list of {time, func, cell_uid} dicts.
    path = Path(path)
    if not path.exists():
        return []
    df = pd.read_csv(path)
    if "cell_uid" in df.columns:
        df["cell_uid"] = pd.to_numeric(df["cell_uid"], errors="coerce")
    return df.to_dict("records")

## Drawing — 2-D flat sheet (divisions / deaths), faces by cell state

In [ ]:
def _flat_face_color(sheet):
    # (Nf, 4) RGBA: normal grey, dividing magenta, dying/extruding black.
    return np.array([
        mcolors.to_rgba(FLAT_TYPE_COLORS.get(ct, FLAT_TYPE_COLORS["normal"]))
        for ct in sheet.face_df["cell_type"]
    ])


def _flat_legend_handles():
    return [Patch(facecolor=FLAT_TYPE_COLORS[k], edgecolor="#808080", label=k)
            for k in ("normal", "dividing", "extruding")]


def _frame_limits(history, times, coords):
    # Fixed (min, max) per-axis limits from the first frame, with a 5% margin.
    sheet0 = history.retrieve(times[0])
    bounds = sheet0.vert_df[coords].describe().loc[["min", "max"]]
    margin = (bounds.loc["max"] - bounds.loc["min"]).max() * 0.05
    return {c: (bounds.loc["min", c] - margin, bounds.loc["max", c] + margin) for c in coords}


def _draw_2d_frame(sheet, title, lims):
    # Draw one flat-sheet frame (faces by cell state). Returns Figure or None.
    from tyssue.draw import sheet_view
    try:
        fig, ax = plt.subplots(figsize=(6.4, 5.0))
        sheet_view(
            sheet, coords=COORDS_2D, ax=ax,
            face={"visible": True, "color": _flat_face_color(sheet), "alpha": 1.0},
            edge={"visible": True, "color": "#808080", "width": 1.0},
        )
        ax.set_aspect("equal")
        ax.set_xlim(*lims["x"]); ax.set_ylim(*lims["y"])
        ax.set_title(title, fontsize=10)
        ax.legend(handles=_flat_legend_handles(), loc="upper left",
                  bbox_to_anchor=(1.01, 1.0), frameon=False, fontsize=8)
    except Exception as exc:  # noqa: BLE001
        print(f"frame {title} failed ({type(exc).__name__}: {exc}); skipping")
        plt.close("all")
        return None
    return fig


def save_gif_2d(history, out_path: Path):
    times = list(history.time_stamps)
    if not times:
        return
    lims = _frame_limits(history, times, COORDS_2D)
    idx = np.unique(np.round(np.linspace(0, len(times) - 1,
                                         min(NUM_GIF_FRAMES, len(times)))).astype(int))
    tmp = Path(tempfile.mkdtemp())
    n = 0
    try:
        for i in idx:
            t = times[int(i)]
            fig = _draw_2d_frame(history.retrieve(t), f"t = {float(t):.1f}", lims)
            if fig is None:
                continue
            fig.savefig(tmp / f"frame_{n:04d}.png", dpi=GIF_DPI, bbox_inches="tight")
            plt.close(fig); n += 1
        if n == 0:
            print(f"no renderable frames for {out_path.name}; skipping GIF"); return
        subprocess.run(["magick", "-delay", "12", "-loop", "0",
                        (tmp / "frame_*.png").as_posix(), str(out_path)], check=True)
        print(f"  wrote {out_path.name} ({n} frames)")
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def save_stills_2d(history, out_dir: Path):
    times = list(history.time_stamps)
    if not times:
        return
    lims = _frame_limits(history, times, COORDS_2D)
    for frac in np.linspace(0.0, 1.0, N_STILLS):
        t = times[int(round(frac * (len(times) - 1)))]
        fig = _draw_2d_frame(history.retrieve(t), f"t = {float(t):.1f}", lims)
        if fig is None:
            continue
        fig.savefig(out_dir / f"still_t{float(t):06.1f}.png", dpi=FIG_DPI, bbox_inches="tight")
        plt.close(fig)

## Drawing — 3-D crypt (gillespie), faces by cell_type

Every frame is rendered ourselves (rather than via `tyssue.create_gif_3d`) because
that helper calls `savefig` outside its per-frame try/except, so a single frame
whose matplotlib-3D projection goes singular (which happens on the crypt after a
division/extrusion reindexes the mesh) aborts the whole GIF. Here each frame is
guarded, so a bad frame is skipped and the rest of the animation survives.

In [ ]:
def _draw_3d_frame(sheet, lims, title, figsize=None):
    # Draw one crypt frame (faces by cell_type) into a fresh Axes3D. Returns Figure or None.
    from tyssue import config
    from tyssue.draw.plt_draw import (
        sheet_view_3d, patch_2d_collections_to_3d, _auto_tick_fontsize_3d,
    )
    from vivarium_tyssue.draw import crypt_cell_type_kwds, CELL_TYPE_COLORS

    ds = config.draw.sheet_spec()
    ds["face"]["visible"] = True
    ds["face"]["alpha"] = 1.0
    # Grey (not black) edges so the black "extruding" faces stay distinguishable.
    ds["edge"]["color"] = "#808080"
    ds["face"]["color"] = crypt_cell_type_kwds(sheet)["face"]["color"]
    try:
        fig = plt.figure(figsize=figsize)
        ax = fig.add_subplot(111, projection="3d")
        ax.view_init(elev=30, azim=45)
        fig, ax = sheet_view_3d(
            sheet, coords=COORDS_3D, ax=ax,
            legend=CELL_TYPE_COLORS, cull_back_edges=True, **ds,
        )
        patch_2d_collections_to_3d(ax)
        ax.set(xlim=lims["x"], ylim=lims["y"], zlim=lims["z"])
        # Recompute the box aspect from the FIXED limits (sheet_view_3d set it from
        # this frame's auto-scaled extent), else the crypt's long z-axis drifts frame
        # to frame as divisions/extrusions shift the per-frame z-extent.
        ax.set_box_aspect((lims["x"][1] - lims["x"][0],
                           lims["y"][1] - lims["y"][0],
                           lims["z"][1] - lims["z"][0]))
        # Same reason for the tick-label font (sheet_view_3d sized it per-frame).
        _auto_tick_fontsize_3d(ax, base_size=8, min_size=4)
        ax.set_title(title, fontsize=9)
        # Force the (occasionally singular) projection now, inside the guard.
        fig.canvas.draw()
    except Exception as exc:  # noqa: BLE001
        print(f"frame {title} failed ({type(exc).__name__}: {exc}); skipping")
        plt.close("all")
        return None
    return fig


def save_gif_3d(history, out_path: Path):
    times = list(history.time_stamps)
    if not times:
        return
    lims = _frame_limits(history, times, COORDS_3D)
    idx = np.unique(np.round(np.linspace(0, len(times) - 1,
                                         min(NUM_GIF_FRAMES, len(times)))).astype(int))
    tmp = Path(tempfile.mkdtemp())
    n = 0
    try:
        for i in idx:
            t = times[int(i)]
            fig = _draw_3d_frame(history.retrieve(t), lims, f"t = {float(t):.2f}", figsize=(5.0, 8.0))
            if fig is None:
                continue
            fig.savefig(tmp / f"frame_{n:04d}.png", dpi=GIF_DPI)
            plt.close(fig); n += 1
        if n == 0:
            print(f"no renderable frames for {out_path.name}; skipping GIF"); return
        subprocess.run(["magick", "-delay", "12", "-loop", "0",
                        (tmp / "frame_*.png").as_posix(), str(out_path)], check=True)
        print(f"  wrote {out_path.name} ({n} frames)")
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def save_stills_3d(history, out_dir: Path):
    times = list(history.time_stamps)
    if not times:
        return
    lims = _frame_limits(history, times, COORDS_3D)
    for frac in np.linspace(0.0, 1.0, N_STILLS):
        t = times[int(round(frac * (len(times) - 1)))]
        # Fixed figsize (as the gif) and NO bbox_inches="tight": tight-cropping would
        # resize each still to its own content, so the crypt's z-axis would look a
        # different length per snapshot. Constant figsize + box_aspect keep it stable.
        fig = _draw_3d_frame(history.retrieve(t), lims, f"t = {float(t):.2f}", figsize=(5.0, 8.0))
        if fig is None:
            continue
        fig.savefig(out_dir / f"still_t{float(t):06.2f}.png", dpi=FIG_DPI)
        plt.close(fig)

## Analysis (gillespie)

In [ ]:
def _type_palette():
    from vivarium_tyssue.draw import CELL_TYPE_COLORS
    return CELL_TYPE_COLORS


def cell_type_over_time(history) -> pd.DataFrame:
    # Count of each cell type at every recorded timepoint (wide: time x types).
    face = history.datasets["face"]
    df = face[face["is_alive"] > 0] if "is_alive" in face.columns else face
    counts = df.groupby(["time", "cell_type"]).size().unstack(fill_value=0)
    counts = counts.reindex(sorted(counts.index)).reset_index()
    return counts


def cell_type_along_z(history, nbins: int = Z_NBINS_CT) -> pd.DataFrame:
    # Mean count of each cell type per z-bin, averaged over all timepoints.
    face = history.datasets["face"]
    df = (face[face["is_alive"] > 0] if "is_alive" in face.columns else face).copy()
    z = df["z"].astype(float)
    bins = np.linspace(z.min(), z.max(), nbins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])
    df["zbin"] = pd.cut(z, bins, include_lowest=True, labels=centers)
    n_frames = df["time"].nunique()
    g = df.groupby(["zbin", "cell_type"], observed=True).size().reset_index(name="count")
    g["count"] = g["count"] / max(n_frames, 1)          # per-frame mean occupancy
    g["zcenter"] = g["zbin"].astype(float)
    return g[["zcenter", "cell_type", "count"]]


def events_along_z(history, events: list, nbins: int = Z_NBINS) -> pd.DataFrame:
    # Count of each event type per z-bin. Each event's z is the mean z of its cell
    # (by unique_id) across the recorded history.
    face = history.datasets["face"]
    z_by_uid = face.groupby("unique_id")["z"].mean()
    z_all = face["z"].astype(float)
    bins = np.linspace(z_all.min(), z_all.max(), nbins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])

    rows = []
    for e in events:
        uid = e["cell_uid"]
        if uid is None or (isinstance(uid, float) and np.isnan(uid)):
            continue
        uid = int(uid)
        if uid not in z_by_uid.index:
            continue
        rows.append({"func": e["func"], "z": float(z_by_uid.loc[uid])})
    if not rows:
        return pd.DataFrame(columns=["zcenter", "func", "count"])
    edf = pd.DataFrame(rows)
    edf["zbin"] = pd.cut(edf["z"], bins, include_lowest=True, labels=centers)
    g = edf.groupby(["zbin", "func"], observed=True).size().reset_index(name="count")
    g["zcenter"] = g["zbin"].astype(float)
    return g[["zcenter", "func", "count"]]

## Analysis plots  (legends placed outside the axes to avoid overlap)

In [ ]:
EVENT_LABELS = {
    "division": "division",
    "differentiation": "differentiation",
    "apoptosis_extrusion": "extrusion (death)",
}
EVENT_COLORS = {
    "division": "#C71FE0",
    "differentiation": "#0072B2",
    "apoptosis_extrusion": "#000000",
}


def _legend_outside(ax, **kw):
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=False, fontsize=8, **kw)


def plot_cell_type_over_time(counts: pd.DataFrame, out_path: Path):
    palette = _type_palette()
    types = [c for c in CELL_TYPE_ORDER if c in counts.columns]
    fig, ax = plt.subplots(figsize=(7.6, 4.2))
    ax.stackplot(
        counts["time"], *[counts[t].to_numpy() for t in types],
        labels=types, colors=[palette.get(t, "#999999") for t in types], alpha=0.9,
    )
    ax.set_xlabel("time"); ax.set_ylabel("cell count")
    ax.set_title("Gillespie crypt: cell-type distribution over time", fontsize=11)
    ax.margins(x=0); _legend_outside(ax)
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight"); plt.show()


def plot_cell_type_along_z(g: pd.DataFrame, out_path: Path):
    # One line per (biological) cell type: mean occupancy vs z.
    palette = _type_palette()
    wide = g.pivot(index="zcenter", columns="cell_type", values="count").fillna(0.0).sort_index()
    types = [c for c in BIO_TYPES if c in wide.columns]
    fig, ax = plt.subplots(figsize=(7.6, 4.2))
    z = wide.index.to_numpy()
    for t in types:
        ax.plot(z, wide[t].to_numpy(), "-", color=palette.get(t, "#999999"), linewidth=1.8, label=t)
    ax.set_xlabel("z position (crypt length)"); ax.set_ylabel("mean cells per z-bin")
    ax.set_title("Gillespie crypt: cell-type spatial distribution along z", fontsize=11)
    ax.margins(x=0); ax.grid(True, alpha=0.25, linewidth=0.6); _legend_outside(ax)
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight"); plt.show()


def plot_events_along_z(g: pd.DataFrame, out_path: Path):
    fig, ax = plt.subplots(figsize=(7.6, 4.2))
    if g.empty:
        ax.text(0.5, 0.5, "no events recorded", ha="center", va="center", transform=ax.transAxes)
    else:
        wide = g.pivot(index="zcenter", columns="func", values="count").fillna(0.0).sort_index()
        funcs = [f for f in EVENT_COLORS if f in wide.columns] + \
                [f for f in wide.columns if f not in EVENT_COLORS]
        z = wide.index.to_numpy()
        width = (z[1] - z[0]) * 0.8 / max(len(funcs), 1) if len(z) > 1 else 0.5
        for i, f in enumerate(funcs):
            ax.bar(z + (i - (len(funcs) - 1) / 2) * width, wide[f].to_numpy(), width=width,
                   color=EVENT_COLORS.get(f, "#999999"), label=EVENT_LABELS.get(f, f), alpha=0.9)
        _legend_outside(ax)
    ax.set_xlabel("z position (crypt length)"); ax.set_ylabel("event count")
    ax.set_title("Gillespie crypt: event-type spatial distribution along z", fontsize=11)
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight"); plt.show()

## Divisions (flat square)

Set `MAKE_GIFS = False` to skip the (slow) animation while iterating on the stills.

In [ ]:
MAKE_GIFS = True

div_dir = OUT_DIR / "divisions"
div_path = div_dir / "history.hf5"
if not div_path.exists():
    print("no divisions archive — run `python discrete_events.py divisions` first")
else:
    hist = LoadedHistory(div_path)
    if MAKE_GIFS:
        save_gif_2d(hist, div_dir / "divisions.gif")
    save_stills_2d(hist, div_dir)
    print("[divisions] done")

## Deaths (flat square)

In [ ]:
death_dir = OUT_DIR / "deaths"
death_path = death_dir / "history.hf5"
if not death_path.exists():
    print("no deaths archive — run `python discrete_events.py deaths` first")
else:
    hist = LoadedHistory(death_path)
    if MAKE_GIFS:
        save_gif_2d(hist, death_dir / "deaths.gif")
    save_stills_2d(hist, death_dir)
    print("[deaths] done")

## Gillespie (3-D crypt) — GIF, stills & analyses

In [ ]:
gil_dir = OUT_DIR / "gillespie"
gil_path = gil_dir / "history.hf5"
if not gil_path.exists():
    print("no gillespie archive — run `python discrete_events.py gillespie` first")
else:
    hist = LoadedHistory(gil_path)
    events = load_events(gil_dir / "events.csv")
    print(f"[gillespie] loaded {len(events)} events")

    if MAKE_GIFS:
        save_gif_3d(hist, gil_dir / "gillespie.gif")
    save_stills_3d(hist, gil_dir)

    counts = cell_type_over_time(hist)
    counts.to_csv(gil_dir / "cell_type_over_time.csv", index=False)
    plot_cell_type_over_time(counts, gil_dir / "cell_type_over_time.png")

    ctz = cell_type_along_z(hist)
    ctz.to_csv(gil_dir / "cell_type_along_z.csv", index=False)
    plot_cell_type_along_z(ctz, gil_dir / "cell_type_along_z.png")

    evz = events_along_z(hist, events)
    evz.to_csv(gil_dir / "events_along_z.csv", index=False)
    plot_events_along_z(evz, gil_dir / "events_along_z.png")
    print("[gillespie] done")